# Generate replayable r0 datasets

Issue #27 setup slice: linear/sparse, periodic and finite Fourier tasks, CPU or GPU, bounded parallel generation, Drive persistence and resume. More families and the embedded explorer remain separate PRs.

Run cells in order. Select a GPU runtime before choosing `cuda`. Installation requires Python 3.12+. This notebook uses only the generator's NumPy/Torch dependencies, not the evaluation stack. A Colab runtime may require a restart after installation. All outputs below are unexecuted in this committed notebook.


In [ ]:
import subprocess
import sys

assert sys.version_info >= (3, 12), "Select a Python 3.12+ runtime"
REVISION = "6f63fcb1241a101a874e6e3bea9a0097a1567258"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy==2.5.5", "torch==2.15.0"], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        f"git+https://github.com/Avistian/r0.git@{REVISION}",
    ],
    check=True,
)

## Persistent output
Mount Drive once. Use one notebook coordinator per output directory; Drive does not provide a distributed writer lock. Keep the same recipe, seed and limits to resume. Change the output directory when changing a request. Interrupted groups stay incomplete; failed tasks require `retry_failed=True` explicitly.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/r0-generation")
ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
from dataclasses import asdict, replace

import torch

from r0.modules.generation import RunRequest, run_guarded
from r0.modules.priors import FourierRecipe, LinearRecipe

DEVICE = "cpu"  # Change to "cuda" after selecting a GPU runtime.
assert DEVICE in ("cpu", "cuda")
assert DEVICE != "cuda" or torch.cuda.is_available(), "CUDA unavailable; no fallback"
print(
    {
        "source": REVISION,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu",
    }
)
FAMILY = "linear"  # "linear", "rff", or "periodic"
assert FAMILY in ("linear", "rff", "periodic")
controls = LinearRecipe(rows=128, features=8, context_rows=96, noise_ratio=0.5)
recipe = (
    controls
    if FAMILY == "linear"
    else FourierRecipe(task=controls, family=FAMILY, terms=256 if FAMILY == "rff" else 16)
)
smoke_request = RunRequest(
    recipe, seed=27, count=8, group_size=4, max_seconds=60, max_bytes=10 * 1024**2, device=DEVICE
)
smoke = run_guarded(smoke_request, ROOT / f"smoke-{FAMILY}-{DEVICE}", workers=2)
print(json.dumps(asdict(smoke), indent=2))
assert smoke.returncode == 0, smoke.stderr

## Estimate before scaling
The report's `generated` count includes successful computations, including any that could not be committed. `completed` counts all verified saved tasks, including earlier invocations. Time includes generation, serialization and persistence, but excludes interpreter startup; `elapsed_seconds` includes startup. Peak RSS is the child process lifetime peak; GPU memory is PyTorch's allocated peak, not total device memory. A killed child has no completed report: do not infer zero failures or memory use.

A fresh smoke run retains its measurement next to its output directory. On resume the notebook loads that measurement and labels it historical; it does not treat skipped tasks as a new timing. If hardware changes, use a fresh smoke directory and measurement before estimating new workloads. Estimates apply only to this recipe/hardware and are not measured 100M-task performance.


In [ ]:
COUNT = 1000  # Explicitly choose the desired task count, up to 100_000_000.
MAX_SECONDS = 300
MAX_BYTES = 100 * 1024**2
WORKERS = 2
summary = smoke.summary
measurement_path = ROOT / f"smoke-{FAMILY}-{DEVICE}-measurement.json"
identity = json.loads(json.dumps({"request": asdict(smoke_request), "source": REVISION}))
if summary["generated"] > 0:
    measurement_path.write_text(json.dumps({"identity": identity, "summary": summary}))
else:
    saved = json.loads(measurement_path.read_text())
    assert saved["identity"] == identity, "Measurement belongs to a different request"
    summary = saved["summary"]
    print("Using retained measurement from earlier smoke; not a new timing:", summary)
rate = summary["generated_per_second"]
print(
    {
        "measured_tasks_per_second": rate,
        "estimated_seconds": COUNT / rate,
        "estimate_scope": "same recipe and hardware; storage and larger-run overhead may differ",
    }
)
large_request = replace(smoke_request, count=COUNT, max_seconds=MAX_SECONDS, max_bytes=MAX_BYTES)
print(json.dumps(asdict(large_request), indent=2))

In [ ]:
RUN_LARGER = False  # Set True only after reviewing the estimate and limits above.
if RUN_LARGER:
    large = run_guarded(
        large_request, ROOT / f"large-{FAMILY}-{DEVICE}", workers=WORKERS, retry_failed=False
    )
    print(json.dumps(asdict(large), indent=2))

## Verify and export
ZIP groups contain lossless predictor/truth arrays and JSON cards. Hidden truth is separate from predictor inputs. Verify each completed archive before using it. Drive keeps the groups for resume or download; downloading below exports one bounded group, not the entire large run. Throughput and Colab durability still need measurement on your selected runtime.


In [ ]:
from r0.modules.generation import verify_group

archives = sorted((ROOT / f"smoke-{FAMILY}-{DEVICE}").glob("groups/*/*/data-*.zip"))
for archive in archives:
    if archive.with_suffix(".json").exists():
        print(archive.name, verify_group(archive))
# Optional: files.download(str(archives[0]))